In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
from tqdm import tqdm
import matplotlib.pyplot as plt

# === Hiperparâmetros ===
STATE_SIZE = 1
ACTION_SIZE = 1
GAMMA = 0.99
ACTOR_LR = 1e-4
CRITIC_LR = 1e-3
BATCH_SIZE = 64
MEMORY_SIZE = 100000
TAU = 0.005
MAX_STEPS = 200
MAX_EPISODES = 1000
ACTION_BOUND = 2.0
ACTION_EFFECT = 80
NOISE_STD = 0.05
rewards_history = []
erro_final_por_ep = []

# === A2C/A3C específicos ===
ENTROPY_COEF = 0.01        # incentivo à exploração
VALUE_COEF = 0.5           # peso da loss do crítico
N_STEPS_RETURN = 5         # n-step advantage
NUM_WORKERS = 1            # defina >1 para emular A3C (sequencial)

# === Replay Buffer (não usado por A2C/A3C; mantido por compatibilidade) ===
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))
    def __len__(self):
        return len(self.buffer)

# === Redes do A2C/A3C ===
class Actor(nn.Module):
    def __init__(self, state_size, action_size):
        super(Actor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        self.mean_head = nn.Linear(64, action_size)
        self.log_std = nn.Parameter(torch.zeros(action_size))  # política Gaussiana contínua

    def forward(self, x):
        h = self.net(x)
        mean = self.mean_head(h)
        log_std = self.log_std.expand_as(mean)
        std = torch.exp(log_std)
        return mean, std

    def sample_action(self, x):
        mean, std = self.forward(x)
        normal = torch.distributions.Normal(mean, std)
        z = normal.sample()
        action = torch.tanh(z) * ACTION_BOUND
        # log_prob com correção do tanh
        log_prob = normal.log_prob(z) - torch.log(1 - torch.tanh(z).pow(2) + 1e-6)
        return action, log_prob.sum(dim=1, keepdim=True)

    def act_deterministic(self, x):
        mean, _ = self.forward(x)
        return torch.tanh(mean) * ACTION_BOUND

class Critic(nn.Module):
    def __init__(self, state_size, action_size):
        super(Critic, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, state, action=None):
        return self.fc(state)

# === Inicialização ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

actor = Actor(STATE_SIZE, ACTION_SIZE).to(device)
critic = Critic(STATE_SIZE, ACTION_SIZE).to(device)

# “targets” mantidos por compatibilidade (não usados em A2C/A3C)
target_actor = Actor(STATE_SIZE, ACTION_SIZE).to(device)
target_critic = Critic(STATE_SIZE, ACTION_SIZE).to(device)
target_actor.load_state_dict(actor.state_dict())
target_critic.load_state_dict(critic.state_dict())

actor_optimizer = optim.Adam(actor.parameters(), lr=ACTOR_LR)
critic_optimizer = optim.Adam(critic.parameters(), lr=CRITIC_LR)

memory = ReplayBuffer(MEMORY_SIZE)
losses = []

# === Funções auxiliares ===
def select_action(state, noise=NOISE_STD):
    state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
    with torch.no_grad():
        action, _ = actor.sample_action(state_t)
    return action.cpu().numpy().flatten()

def soft_update(target, source, tau):
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)

def calculate_reward(erro_lateral):
    normalized_error = abs(erro_lateral) / 100
    reward = 1.0 - normalized_error**2
    return reward

def simulate_step(state, action):
    erro_lateral = state[0]
    erro_lateral += float(action) * ACTION_EFFECT
    erro_lateral += random.uniform(-5, 5)
    erro_lateral = np.clip(erro_lateral, -100, 100)
    return np.array([erro_lateral])

# === Utilidades para retorno n-step e vantagens ===
def compute_nstep_returns(rewards, values, dones, gamma=GAMMA):
    T = len(rewards)
    returns = torch.zeros_like(values)
    running_return = torch.zeros(1, device=device)
    for t in reversed(range(T)):
        running_return = rewards[t] + gamma * running_return * (1.0 - dones[t])
        returns[t] = running_return
    advantages = returns - values
    return advantages.detach(), returns.detach()

# === Treino A2C por episódio ===
def a2c_update(trajectory):
    states = torch.cat([t['state'] for t in trajectory], dim=0)
    actions = torch.cat([t['action'] for t in trajectory], dim=0)
    log_probs = torch.cat([t['log_prob'] for t in trajectory], dim=0)
    rewards = torch.cat([t['reward'] for t in trajectory], dim=0)
    dones = torch.cat([t['done'] for t in trajectory], dim=0)
    values = torch.cat([t['value'] for t in trajectory], dim=0) # These values now have requires_grad=True

    # Compute n-step returns
    # The 'values' passed here should be detached to ensure 'returns' are treated as targets
    _, returns = compute_nstep_returns(rewards, values.detach(), dones, gamma=GAMMA)

    # Advantages for actor: G_t - V(s_t)
    # 'returns' are detached, 'values' (from trajectory) have requires_grad=True
    advantages = returns - values

    # Actor loss: -E[log_prob * advantage] - entropy
    # 'log_probs' now has requires_grad=True. 'advantages' are detached for actor loss.
    entropy = -(log_probs).mean()  # aproximação da entropia com log_prob (para gaussiana com tanh é um proxy)
    actor_loss = -(log_probs * advantages.detach()).mean() - ENTROPY_COEF * entropy

    # Critic loss: MSE entre V(s) e returns
    # 'values' (from trajectory) have requires_grad=True, 'returns' are detached targets
    critic_loss = nn.MSELoss()(values, returns)

    # Otimizações
    actor_optimizer.zero_grad()
    actor_loss.backward()
    actor_optimizer.step()

    critic_optimizer.zero_grad()
    critic_loss.backward()
    critic_optimizer.step()

    return actor_loss.item(), critic_loss.item()

# === Loop de interação (A2C) ===
for episode in tqdm(range(MAX_EPISODES), desc="Treinando episódios (A2C)"):
    state = np.array([random.uniform(-100, 100)])
    total_reward = 0
    trajectory = []

    for step in range(MAX_STEPS):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        # Removed with torch.no_grad() to allow gradients for actor's log_prob and critic's value
        action_t, logp_t = actor.sample_action(state_t)
        value_t = critic(state_t)

        action = action_t.cpu().numpy().flatten()
        next_state = simulate_step(state, action)
        erro_lateral = next_state[0]

        reward = calculate_reward(erro_lateral)
        done = abs(erro_lateral) > 100

        trajectory.append({
            'state': state_t,
            'action': torch.FloatTensor(action).unsqueeze(0).to(device),
            'log_prob': logp_t,
            'reward': torch.FloatTensor([reward]).to(device),
            'done': torch.FloatTensor([float(done)]).to(device),
            'value': value_t
        })

        state = next_state
        total_reward += reward

        if done:
            break

    # Atualiza com a trajetória do episódio
    a_loss, c_loss = a2c_update(trajectory)
    losses.append((c_loss, a_loss))

    # Atualização suave (mantida por compatibilidade, não necessária)
    soft_update(target_actor, actor, TAU)
    soft_update(target_critic, critic, TAU)

    rewards_history.append(total_reward)
    erro_final_por_ep.append(state[0])
    print(f"[Ep {episode}] Reward: {total_reward:.2f} | Erro final: {state[0]:.2f}")

# === Salvar os modelos treinados ===
torch.save(actor.state_dict(), "a2c_actor.pth")
torch.save(critic.state_dict(), "a2c_critic.pth")
print("✅ Modelos salvos (a2c_actor.pth, a2c_critic.pth)")

# === Gráfico de recompensas ===
plt.figure(figsize=(10, 5))
plt.plot(rewards_history, label="Recompensa por episódio")
plt.xlabel("Episódio")
plt.ylabel("Recompensa total")
plt.title("Evolução da recompensa (A2C)")
plt.grid()
plt.legend()
plt.show()

# === Gráfico de erro final ===
plt.figure(figsize=(10, 5))
plt.plot(erro_final_por_ep, label="Erro lateral final")
plt.xlabel("Episódio")
plt.ylabel("Erro final")
plt.title("Convergência do erro lateral ao longo dos episódios (A2C)")
plt.grid()
plt.legend()
plt.show()


In [ ]:

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd

# === Hiperparâmetros ===
STATE_SIZE = 1
ACTION_SIZE = 1
ACTION_LIMIT = 0.9
ERRO_MAX = 300.0  # usado para normalização e visualização

# === Rede Actor (mesma usada no treino A2C) ===
class Actor(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=64):
        super(Actor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU()
        )
        self.mean_head = nn.Linear(hidden_size, action_size)
        self.log_std = nn.Parameter(torch.zeros(action_size))  # política Gaussiana

    def forward(self, x):
        h = self.net(x)
        mean = self.mean_head(h)
        log_std = self.log_std.expand_as(mean)
        std = torch.exp(log_std)
        return mean, std

    def act_deterministic(self, x):
        mean, _ = self.forward(x)
        return torch.tanh(mean) * ACTION_LIMIT

# === Inicialização ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
actor = Actor(STATE_SIZE, ACTION_SIZE).to(device)

# Carregar modelo salvo (treinado com A2C)
actor.load_state_dict(torch.load("a2c_actor.pth", map_location=device))
actor.eval()
print("✅ Modelo A2C carregado!")

# === Função para escolher ação ===
def select_action(state):
    state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
    with torch.no_grad():
        action = actor.act_deterministic(state_t).cpu().numpy().flatten()
    return action[0]

# === Geração de dados ===
test_states = np.linspace(-ERRO_MAX, ERRO_MAX, 200).reshape(-1, 1)
action_list = [select_action(s) for s in test_states]

# === Exportar CSV ===
pd.DataFrame({
    "Erro_Lateral": test_states.flatten(),
    "Velocidade_Angular": action_list
}).to_csv("velocidade_predita_angular_a2c.csv", index=False)
print("📁 CSV salvo como 'velocidade_predita_angular_a2c.csv'")

# === Plot ===
plt.figure(figsize=(10, 5))
plt.plot(test_states, action_list, color='purple', label="Velocidade Angular (A2C actor)")
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.xlabel("Erro Lateral")
plt.ylabel(f"Velocidade Angular (±{ACTION_LIMIT})")
plt.title("Velocidade Angular vs Erro Lateral (A2C actor)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("velocidade_vs_erro_angular_a2c.png")
plt.show()
print("🖼️ Gráfico salvo como 'velocidade_vs_erro_angular_a2c.png'")